In [2]:
import os
os.chdir('..')
os.getcwd()

'/ClusterApp'

In [3]:
from api.src.service.gnps import Proteosafe

In [4]:
import pandas as pd
import requests
import io

base_url = 'https://br.gnps2.org/resultfile?task='

taskid = '7e136d38c5424eefaaad243d7d033bac'

url_to_features = f'{base_url}{taskid}&file=nf_output/clustering/featuretable_reformated.csv'
url_to_metadata = f'{base_url}{taskid}&file=nf_output/metadata/merged_metadata.tsv'
url_to_edges = f'{base_url}{taskid}&file=nf_output/networking/filtered_pairs.tsv'

feat = pd.read_csv(io.StringIO(requests.get(url_to_features).text))
meta = pd.read_csv(io.StringIO(requests.get(url_to_metadata).text), sep='\t')
net = pd.read_csv(io.StringIO(requests.get(url_to_edges).text), sep='\t')

In [5]:
feat.head()

,row ID,row m/z,row retention time,intensity_measure,Baat132GB.mzXML Peak area,Baat143GB.mzXML Peak area,Baat126GB.mzXML Peak area,Baat132B.mzXML Peak area,Baat126B.mzXML Peak area,Baat143C.mzXML Peak area,...,Baat214Du.mzXML Peak area,Baat126Ce.mzXML Peak area,Baat176Du.mzXML Peak area,Baat196Du.mzXML Peak area,Baat192Du.mzXML Peak area,Baat209Du.mzXML Peak area,Baat194Du.mzXML Peak area,Baat180Du.mzXML Peak area,Baat217Du.mzXML Peak area,Baat153B_20210816153354.mzXML Peak area
0,1,432.237281,6.670470,area,67163.871102,45591.376960,40999.377936,65978.967426,46447.795992,22335.234423,...,2018.390799,23994.443665,12345.648112,10125.165367,11086.577320,7516.681638,12653.607698,8595.526617,1237.697820,50141.352470
1,2,460.268697,6.669183,area,53792.832312,40222.626160,26193.910952,51358.367843,35979.519731,17849.168025,...,3414.771765,11343.435644,11517.450625,6392.857985,12088.405257,9815.969434,6631.494340,7622.822367,1935.933407,12955.545460
2,3,520.339179,7.349832,area,55546.085427,46677.369172,12888.261801,6315.507060,7014.187592,59475.609024,...,31650.685396,19626.053580,41682.120104,44454.935130,33414.410019,33024.603849,37536.215100,42520.402836,31292.076211,6800.599878
3,4,533.324933,5.626430,area,248040.432660,0.000000,0.000000,0.000000,0.000000,2984.805930,...,0.000000,220.692910,0.000000,107565.895689,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,5,415.210732,6.669842,area,28656.749205,19721.509158,17465.438864,27341.642149,20600.125733,9693.589112,...,1194.153304,9901.454113,5248.669541,4697.603521,4750.768231,3542.498087,5562.553665,3733.495621,780.017779,20733.160661


In [9]:
net.head()

,CLUSTERID1,CLUSTERID2,ComponentIndex,Cosine,DeltaMZ
0,6504,6790,1,0.9832,97.326
1,6790,10183,1,0.9833,-0.003
2,4310,6790,1,0.9809,-0.299
3,3225,5887,1,0.9881,-12.809
4,3225,6145,1,0.9883,8.593


In [11]:
feat.shape

(10796, 285)

In [12]:
feat['row ID'].isin(net['CLUSTERID1'].tolist()+net['CLUSTERID2'].tolist()).sum()

6520

In [13]:
feat = feat[feat['row ID'].isin(net['CLUSTERID1'].tolist()+net['CLUSTERID2'].tolist())]
feat.shape

(6520, 285)

In [6]:
meta.head()

,filename,ATTRIBUTE_sample_name,ATTRIBUTE_sex,ATTRIBUTE_cage_id,ATTRIBUTE_tissue_type,ATTRIBUTE_female_parent,ATTRIBUTE_female_parent_genotype,ATTRIBUTE_genotype,ATTRIBUTE_male_parent,ATTRIBUTE_male_parent_genotype,ATTRIBUTE_plate number
0,Baat126B.mzXML,Baat126B,male,F1,brain,baat7,het,ko,baat10,het,2
1,Baat126C.mzXML,Baat126C,male,F1,colon,baat7,het,ko,baat10,het,2
2,Baat126Ce.mzXML,Baat126Ce,male,F1,cecum,baat7,het,ko,baat10,het,2
3,Baat126GB.mzXML,Baat126GB,male,F1,gall bladder,baat7,het,ko,baat10,het,2
4,Baat126IL.mzXML,Baat126IL,male,F1,ileum,baat7,het,ko,baat10,het,2


In [14]:
feat.apply(lambda a: '{0}_{1}'.format(round(a['row m/z']), round(a['row retention time']*60)), axis=1)

0        432_400
1        460_400
2        520_441
3        533_338
4        415_400
          ...   
10790    554_162
10791    712_294
10792    531_179
10793    625_310
10795    531_162
Length: 6520, dtype: object

In [31]:
feat2 = feat[feat.columns[feat.columns.str.contains('Peak area')]].T
feat2

,0,1,2,3,4,5,7,8,10,11,...,10783,10785,10786,10787,10789,10790,10791,10792,10793,10795
Baat132GB.mzXML Peak area,67163.871102,53792.832312,55546.085427,248040.43266,28656.749205,26019.285867,24869.828593,13450.091683,7895.241933,5696.469625,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat143GB.mzXML Peak area,45591.376960,40222.626160,46677.369172,0.00000,19721.509158,15021.222138,15153.590571,15683.058277,3329.217915,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat126GB.mzXML Peak area,40999.377936,26193.910952,12888.261801,0.00000,17465.438864,3723.941069,1367.431348,4643.711352,316.319001,1866.999787,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat132B.mzXML Peak area,65978.967426,51358.367843,6315.507060,0.00000,27341.642149,11725.412129,38470.455902,89324.682637,23501.683468,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat126B.mzXML Peak area,46447.795992,35979.519731,7014.187592,0.00000,20600.125733,14815.818608,42186.386266,95146.819944,24556.664664,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Baat209Du.mzXML Peak area,7516.681638,9815.969434,33024.603849,0.00000,3542.498087,29634.442940,7148.513078,20322.811550,3103.918982,0.000000,...,119.802443,19.709376,13.512850,12.575679,11.095870,7.068965,6.972890,0.000000,0.000000,0.000000
Baat194Du.mzXML Peak area,12653.607698,6631.494340,37536.215100,0.00000,5562.553665,46807.995764,3576.192299,47012.523029,1099.575556,0.000000,...,209.919103,45.383240,13.436601,0.000000,0.000000,208.208931,0.000000,626.559373,13.341527,13.543556
Baat180Du.mzXML Peak area,8595.526617,7622.822367,42520.402836,0.00000,3733.495621,21474.108087,0.000000,26894.682142,0.000000,0.000000,...,419.094327,114.274642,0.000000,27.014642,37.301914,0.000000,13.069040,1278.331048,43.080268,168.093168
Baat217Du.mzXML Peak area,1237.697820,1935.933407,31292.076211,0.00000,780.017779,11177.387997,0.000000,10565.137002,0.000000,0.000000,...,136.376776,72.487408,12.971717,30.255860,8.342159,13.406999,20.851803,874.922312,12.746027,33.955009


In [32]:
feat2.columns = feat.apply(lambda a: '{0}_{1}'.format(round(a['row m/z']), round(a['row retention time']*60)), axis=1).tolist()
feat2 = feat2.reset_index()
feat2['index'] = feat2['index'].str.replace(' Peak area', '')
feat2.set_index('index', inplace=True)
feat2

,432_400,460_400,520_441,533_338,415_400,524_492,568_438,522_465,526_437,496_448,...,532_280,546_266,1097_307,1075_291,659_295,554_162,712_294,531_179,625_310,531_162
index,,,,,,,,,,,,,,,,,,,,,
Baat132GB.mzXML,67163.871102,53792.832312,55546.085427,248040.43266,28656.749205,26019.285867,24869.828593,13450.091683,7895.241933,5696.469625,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat143GB.mzXML,45591.376960,40222.626160,46677.369172,0.00000,19721.509158,15021.222138,15153.590571,15683.058277,3329.217915,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat126GB.mzXML,40999.377936,26193.910952,12888.261801,0.00000,17465.438864,3723.941069,1367.431348,4643.711352,316.319001,1866.999787,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat132B.mzXML,65978.967426,51358.367843,6315.507060,0.00000,27341.642149,11725.412129,38470.455902,89324.682637,23501.683468,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Baat126B.mzXML,46447.795992,35979.519731,7014.187592,0.00000,20600.125733,14815.818608,42186.386266,95146.819944,24556.664664,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Baat209Du.mzXML,7516.681638,9815.969434,33024.603849,0.00000,3542.498087,29634.442940,7148.513078,20322.811550,3103.918982,0.000000,...,119.802443,19.709376,13.512850,12.575679,11.095870,7.068965,6.972890,0.000000,0.000000,0.000000
Baat194Du.mzXML,12653.607698,6631.494340,37536.215100,0.00000,5562.553665,46807.995764,3576.192299,47012.523029,1099.575556,0.000000,...,209.919103,45.383240,13.436601,0.000000,0.000000,208.208931,0.000000,626.559373,13.341527,13.543556
Baat180Du.mzXML,8595.526617,7622.822367,42520.402836,0.00000,3733.495621,21474.108087,0.000000,26894.682142,0.000000,0.000000,...,419.094327,114.274642,0.000000,27.014642,37.301914,0.000000,13.069040,1278.331048,43.080268,168.093168


In [33]:
feat2.shape

(281, 6520)

In [34]:
(feat2==0).sum().sum()

1608778

In [35]:
feat2[feat2!=0] = 1
feat2.head()

,432_400,460_400,520_441,533_338,415_400,524_492,568_438,522_465,526_437,496_448,...,532_280,546_266,1097_307,1075_291,659_295,554_162,712_294,531_179,625_310,531_162
index,,,,,,,,,,,,,,,,,,,,,
Baat132GB.mzXML,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Baat143GB.mzXML,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Baat126GB.mzXML,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Baat132B.mzXML,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Baat126B.mzXML,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
meta.head()

,filename,ATTRIBUTE_sample_name,ATTRIBUTE_sex,ATTRIBUTE_cage_id,ATTRIBUTE_tissue_type,ATTRIBUTE_female_parent,ATTRIBUTE_female_parent_genotype,ATTRIBUTE_genotype,ATTRIBUTE_male_parent,ATTRIBUTE_male_parent_genotype,ATTRIBUTE_plate number
0,Baat126B.mzXML,Baat126B,male,F1,brain,baat7,het,ko,baat10,het,2
1,Baat126C.mzXML,Baat126C,male,F1,colon,baat7,het,ko,baat10,het,2
2,Baat126Ce.mzXML,Baat126Ce,male,F1,cecum,baat7,het,ko,baat10,het,2
3,Baat126GB.mzXML,Baat126GB,male,F1,gall bladder,baat7,het,ko,baat10,het,2
4,Baat126IL.mzXML,Baat126IL,male,F1,ileum,baat7,het,ko,baat10,het,2


In [36]:
(feat2==1).sum().sum()

223342

In [37]:
281*6520

1832120

In [38]:
223342+1608778

1832120

In [39]:
1608778/1832120

0.8780964128987184

In [19]:
meta.shape

(281, 11)

In [40]:
feat2 = feat2.reset_index()

In [41]:
pd.merge(meta, feat2, left_on='filename', right_on='index')

,filename,ATTRIBUTE_sample_name,ATTRIBUTE_sex,ATTRIBUTE_cage_id,ATTRIBUTE_tissue_type,ATTRIBUTE_female_parent,ATTRIBUTE_female_parent_genotype,ATTRIBUTE_genotype,ATTRIBUTE_male_parent,ATTRIBUTE_male_parent_genotype,...,532_280,546_266,1097_307,1075_291,659_295,554_162,712_294,531_179,625_310,531_162
0,Baat126B.mzXML,Baat126B,male,F1,brain,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Baat126C.mzXML,Baat126C,male,F1,colon,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Baat126Ce.mzXML,Baat126Ce,male,F1,cecum,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Baat126GB.mzXML,Baat126GB,male,F1,gall bladder,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Baat126IL.mzXML,Baat126IL,male,F1,ileum,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,Blank26.mzXML,Blank26,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
274,Blank27.mzXML,Blank27,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
275,Blank28.mzXML,Blank28,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
276,Blank29.mzXML,Blank29,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
pd.merge(meta, feat2, left_on='filename', right_on='index').shape

(278, 6532)

In [43]:
pd.merge(meta, feat2, left_on='filename', right_on='index').drop(['index'], axis=1)

,filename,ATTRIBUTE_sample_name,ATTRIBUTE_sex,ATTRIBUTE_cage_id,ATTRIBUTE_tissue_type,ATTRIBUTE_female_parent,ATTRIBUTE_female_parent_genotype,ATTRIBUTE_genotype,ATTRIBUTE_male_parent,ATTRIBUTE_male_parent_genotype,...,532_280,546_266,1097_307,1075_291,659_295,554_162,712_294,531_179,625_310,531_162
0,Baat126B.mzXML,Baat126B,male,F1,brain,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Baat126C.mzXML,Baat126C,male,F1,colon,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Baat126Ce.mzXML,Baat126Ce,male,F1,cecum,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Baat126GB.mzXML,Baat126GB,male,F1,gall bladder,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Baat126IL.mzXML,Baat126IL,male,F1,ileum,baat7,het,ko,baat10,het,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,Blank26.mzXML,Blank26,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
274,Blank27.mzXML,Blank27,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
275,Blank28.mzXML,Blank28,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
276,Blank29.mzXML,Blank29,Blank,Blank,Blank,Blank,Blank,Blank,Blank,Blank,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
pd.merge(meta, feat2, left_on='filename', right_on='index').drop(['index'], axis=1).to_csv('baat_binary.csv', index=None)